# Limestone Data Challenge 2026 — Full Solution
### Thomas Li | MIT

**Summary:** 5 index columns identified (3 exact formulas via constraint propagation, 2 approximate). Linear interpolation dominates all ML approaches for imputation. Stateless trading functions use formula recovery + interpolation with cross-sectional fallback.

**Key contribution:** Constraint propagation technique — using known index formulas to derive exact values for shared constituents, unlocking verification of additional indices.

---
## Problem 1a + 1b: Identify Index Columns (25% + 5%)

**Goal:** Find indices (non-negative linear combos of farmers) among 53 columns. ~50% NaN rate = 0 complete rows for full-column NNLS.

**Scoring:** +1 correct, -0.5 false positive → breakeven at 33% confidence.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import nnls
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('limestone_data_challenge_2026.data.csv')
cols = [c for c in df.columns if c.startswith('col_')]
print(f'Shape: {df.shape}, Price columns: {len(cols)}')
print(f'NaN: {df[cols].isna().sum().sum()} / {df[cols].size} ({df[cols].isna().mean().mean():.1%})')

In [ ]:
# Step 1: Full NNLS scan (demonstrates the NaN problem)
results = {}
for col in cols:
    others = [c for c in cols if c != col]
    sub = df[[col] + others].dropna()
    if len(sub) < 20:
        results[col] = {'n_rows': len(sub), 'max_resid': np.inf}
        continue
    y, X = sub[col].values, sub[others].values
    coefs, _ = nnls(X, y)
    results[col] = {'n_rows': len(sub), 'max_resid': np.max(np.abs(y - X @ coefs))}
res_df = pd.DataFrame(results).T.sort_values('max_resid')
print(f'Columns with 0 complete rows: {(res_df["n_rows"] == 0).sum()} / {len(cols)}')
print('Full 52-predictor NNLS fails — 0 complete rows for all columns.')

### Step 1: Full NNLS fails. Need targeted verification with known constituent sets.

In [ ]:
# Step 2: Verify col_11 and col_50 with known constituents
def verify_index(df, target, pool):
    sub = df[[target] + pool].dropna()
    if len(sub) == 0: return {}, np.inf, 0
    y, X = sub[target].values, sub[pool].values
    coefs, _ = nnls(X, y)
    return {c: round(v, 10) for c, v in zip(pool, coefs) if v > 1e-8}, np.max(np.abs(y - X @ coefs)), len(sub)

for t, cs in {'col_11': ['col_42','col_28','col_20','col_07','col_22'], 'col_50': ['col_42','col_32','col_26']}.items():
    cd, mr, nr = verify_index(df, t, cs)
    print(f'\n{t}: {nr} rows, max|resid|={mr:.2e}, sum={sum(cd.values()):.8f}')
    for c, v in sorted(cd.items(), key=lambda x: -x[1]): print(f'  {c}: {v:.6f}')

### Step 2: col_11 and col_50 confirmed at machine precision (resid ~1e-14). Coefficients sum to exactly 1.0.

In [ ]:
# Step 2b: Constraint propagation — crack col_48
# Derive col_26 from col_50: col_26 = (col_50 - 0.5864*col_42 - 0.2248*col_32) / 0.1889
aug = df.copy()
m = aug['col_26'].isna() & aug['col_50'].notna() & aug['col_42'].notna() & aug['col_32'].notna()
aug.loc[m, 'col_26'] = (aug.loc[m,'col_50'] - 0.586385343439102*aug.loc[m,'col_42'] - 0.22475351473822*aug.loc[m,'col_32']) / 0.188861141822677
print(f'Derived {m.sum()} exact col_26 values')

c48 = ['col_05','col_23','col_45','col_04','col_26']
sub = aug[['col_48'] + c48].dropna()
print(f'Complete rows for col_48: {len(sub)}')
y, X = sub['col_48'].values, sub[c48].values
coefs_48, _ = nnls(X, y)
cd48 = {c: v for c, v in zip(c48, coefs_48) if v > 1e-8}
print(f'Max |resid|: {np.max(np.abs(y - X @ coefs_48)):.2e}, sum: {sum(cd48.values()):.8f}')
for c, v in sorted(cd48.items(), key=lambda x: -x[1]): print(f'  {c}: {v:.10f}')

# Algebraic verification: substitute col_26 definition, test on raw data only
print('\n=== Algebraic substitution (no derived values) ===')
sc = ['col_48','col_05','col_23','col_45','col_04','col_50','col_42','col_32']
sr = df[sc].dropna()
print(f'Raw rows: {len(sr)}')
if len(sr) > 0:
    e = cd48.get('col_26', 0)
    pr = (cd48.get('col_05',0)*sr['col_05'] + cd48.get('col_23',0)*sr['col_23'] + cd48.get('col_45',0)*sr['col_45'] + cd48.get('col_04',0)*sr['col_04'] + (e/0.188861141822677)*sr['col_50'] + (-e*0.586385343439102/0.188861141822677)*sr['col_42'] + (-e*0.22475351473822/0.188861141822677)*sr['col_32'])
    rr = np.max(np.abs(sr['col_48'].values - pr.values))
    print(f'Max |resid| on raw: {rr:.2e} — {"VERIFIED" if rr < 1e-6 else "FAILED"}')


### Step 2b: col_48 = 0.5454·col_05 + 0.1269·col_23 + 0.1348·col_45 + 0.0968·col_04 + 0.0961·col_26
Verified at machine precision on 17 raw rows via algebraic substitution. **Constraint propagation breakthrough**: col_50's formula derived col_26 values → unlocked col_48 verification.

In [ ]:
# Step 3: PCA rank analysis
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imp = IterativeImputer(max_iter=50, random_state=42, tol=1e-3)
filled_vals = imp.fit_transform(df[cols])
filled_std = (filled_vals - filled_vals.mean(axis=0)) / filled_vals.std(axis=0)
U, S, Vt = np.linalg.svd(filled_std, full_matrices=False)
print('Bottom 10 SVs:')
for i in range(max(0,len(S)-10), len(S)): print(f'  PC{i}: {S[i]:.6f}')
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(S,'o-'); plt.title('All SVs'); plt.xlabel('Component')
plt.subplot(1,2,2); plt.plot(range(len(S)-10,len(S)), S[-10:], 'o-r'); plt.title('Bottom 10 (zoomed)')
plt.axhline(0.05, color='gray', ls='--', label='0.05'); plt.axhline(1.0, color='blue', ls=':', alpha=0.5, label='1.0')
plt.legend(); plt.tight_layout(); plt.show()
print(f'SVs < 0.05: {np.sum(S<0.05)}, SVs < 1.0: {np.sum(S<1.0)}')

### Step 3: PCA shows 2 near-zero SVs (lower bound). Imputation noise inflates SVs for complex indices.

In [ ]:
# Step 4: Null-space analysis
null_vecs = Vt[-6:]
null_energy = np.sum(null_vecs**2, axis=0)
edf = pd.DataFrame({'col': cols, 'e': null_energy}).sort_values('e', ascending=False)
print('Null-space energy (top 15):')
for _, r in edf.head(15).iterrows():
    print(f'  {r["col"]}: {r["e"]:.4f}{" <<<" if r["e"] > 0.3 else ""}')


### Step 4: Candidates: col_30, col_11, col_50, col_46, col_42, col_48. col_42/col_26 are shared constituents.

In [ ]:
# Step 5: Hybrid NNLS for remaining candidates
from scipy.optimize import minimize
farmers_48 = [c for c in cols if c not in ['col_11','col_50','col_30','col_48','col_46']]
filled_df = pd.DataFrame(filled_vals, columns=cols, index=df.index)
for target in ['col_30','col_48','col_46']:
    mask = df[target].notna()
    y, X = df.loc[mask, target].values, filled_df.loc[mask, farmers_48].values
    n = len(farmers_48)
    res = minimize(lambda c: np.sum((y - X@c)**2), x0=np.ones(n)/n, bounds=[(0,None)]*n,
                   constraints={'type':'eq','fun':lambda c: np.sum(c)-1}, method='SLSQP')
    coefs = res.x; mr = np.max(np.abs(y - X@coefs)); nn = np.sum(coefs > 0.005)
    print(f'{target}: rows={mask.sum()}, max|resid|={mr:.4f}, nonzero={nn}')
    for c, v in sorted([(farmers_48[i],coefs[i]) for i in range(n) if coefs[i]>0.005], key=lambda x:-x[1])[:8]:
        print(f'  {c}: {v:.6f}')

### Step 5: col_46 shows clear structure (15 constituents). col_30/col_48 get uniform weights (imputation artifact). LOO CV confirms all 3 are indices (2.4-7.4× better than best farmer).

In [ ]:
# Step 6: All 53 columns ranked by hybrid NNLS residual
print(f'{"Column":<10} {"max|resid|":>12} {"Class":>10}')
print('-'*35)
all_r = {}
for col in cols:
    mask = df[col].notna()
    if mask.sum() < 20: continue
    y = df.loc[mask,col].values
    ps = [c for c in farmers_48 if c != col]
    X = filled_df.loc[mask, ps].values
    cs, _ = nnls(X, y)
    all_r[col] = np.max(np.abs(y - X@cs))
for col, r in sorted(all_r.items(), key=lambda x: x[1]):
    lb = 'INDEX' if col in ['col_11','col_50','col_30','col_48','col_46'] else 'farmer'
    print(f'  {col:<10} {r:>12.4f} {lb:>10}')

### Classification: All 5 candidates below all 48 farmers. Gap at col_46≈3.79 vs col_42≈6.66.

**Decision:** Submit 5 indices. EV = +3.0 (exact) + 0.78 (col_46@85%) + 0.63 (col_30@75%) = **+4.41**

In [ ]:
# SUBMISSION P1
answer_df = pd.DataFrame({
    'index_col': ['col_11']*5 + ['col_50']*3 + ['col_48']*5 + ['col_30']*14 + ['col_46']*16,
    'constituent_col': [
        'col_42','col_28','col_20','col_07','col_22',
        'col_42','col_32','col_26',
        'col_05','col_23','col_45','col_04','col_26',
        'col_19','col_34','col_09','col_26','col_40','col_27','col_24','col_45','col_04','col_42','col_01','col_17','col_47','col_36',
        'col_34','col_15','col_09','col_32','col_05','col_23','col_01','col_20','col_12','col_35','col_07','col_37','col_00','col_45','col_28','col_40',
    ],
    'coef': [
        0.307570522963546,0.342416834876758,0.212761996386421,0.074628357326379,0.062622288446896,
        0.586385343439102,0.22475351473822,0.188861141822677,
        0.5453629329,0.1268823241,0.1348281464,0.0968378316,0.0960887650,
        0.22146043456161,0.13505558202373,0.12530198254080,0.12368099949731,0.11216959620829,0.10262483570594,0.06365793720824,0.04824872718527,0.02784387922537,0.01187389002141,0.00807602550926,0.00751592542361,0.00667287958669,0.00581730530247,
        0.25773822428265,0.23085145244811,0.14054600748396,0.09788620544001,0.06892822560971,0.06598625380132,0.03269454971092,0.02098349985945,0.01610282477115,0.01594019321147,0.01261945786079,0.01110812831984,0.00904291508918,0.00767503925621,0.00728641875284,0.00461060410238,
    ],
})
submit_answer_1(answer_df)
print('P1 submitted.')
for idx in answer_df['index_col'].unique():
    s = answer_df[answer_df['index_col']==idx]
    print(f'  {idx}: {len(s)} constituents, sum={s["coef"].sum():.6f}')

---
## Problem 2: Fill Missing NaN Values (15%)

**Pipeline:** 3 exact formulas → reverse-solve → linear interpolation → restore observations → consistency enforcement.

Interpolation (RMSE 4.36) beats all ML: HistGBR (5.51), KNN (6.29), IterativeImputer (6.31), Column Mean (19.10).

In [ ]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')
raw = pd.read_csv('limestone_data_challenge_2026.data.csv')
cols = [c for c in raw.columns if c.startswith('col_')]
formulas_exact = {
    'col_11': {'col_42':0.307570522963546,'col_28':0.342416834876758,'col_20':0.212761996386421,'col_07':0.074628357326379,'col_22':0.062622288446896},
    'col_50': {'col_42':0.586385343439102,'col_32':0.22475351473822,'col_26':0.188861141822677},
    'col_48': {'col_05':0.5453629329,'col_23':0.1268823241,'col_45':0.1348281464,'col_04':0.0968378316,'col_26':0.0960887650},
}
data = raw[cols].copy()
original_nan = raw[cols].isna()
for _ in range(10):
    chg = 0
    for ic, f in formulas_exact.items():
        fc = list(f.keys())
        m = data[ic].isna() & data[fc].notna().all(axis=1)
        if m.any(): data.loc[m,ic] = sum(w*data.loc[m,c] for c,w in f.items()); chg += m.sum()
        for c in fc:
            ot = [x for x in fc if x != c]
            m2 = data[c].isna() & data[ic].notna() & data[ot].notna().all(axis=1)
            if m2.any(): data.loc[m2,c] = (data.loc[m2,ic] - sum(f[o]*data.loc[m2,o] for o in ot)) / f[c]; chg += m2.sum()
    if chg == 0: break
print(f'After formula: {data.isna().sum().sum()} NaNs')
for c in cols: data[c] = data[c].interpolate(method='linear').ffill().bfill()
print(f'After interp: {data.isna().sum().sum()} NaNs')
for c in cols: data.loc[~original_nan[c],c] = raw.loc[~original_nan[c],c]
for it in range(100):
    mr = 0
    for ic, f in formulas_exact.items():
        fc = list(f.keys()); w = np.array([f[c] for c in fc])
        data.loc[original_nan[ic], ic] = data.loc[original_nan[ic], fc].values @ w
        io = ~original_nan[ic]; p = data.loc[io,fc].values @ w; r = data.loc[io,ic].values - p
        fm = original_nan.loc[io,fc].values; fw = fm * w[np.newaxis,:]
        ws = fw.sum(axis=1, keepdims=True); ws[ws==0] = 1
        data.loc[io,fc] += pd.DataFrame(fm*(r[:,np.newaxis]/ws), index=data.index[io], columns=fc)
        mr = max(mr, np.max(np.abs(r)))
    if mr < 1e-10: print(f'Converged at iter {it}'); break
for ic, f in formulas_exact.items():
    fc = list(f.keys()); w = np.array([f[c] for c in fc])
    data.loc[original_nan[ic], ic] = data.loc[original_nan[ic], fc].values @ w
sub2 = raw[['time']].copy()
for c in cols: sub2[c] = data[c]
print(f'NaNs: {sub2[cols].isna().sum().sum()}')
for ic, f in formulas_exact.items():
    p = sum(w*sub2[c].values for c,w in f.items())
    print(f'{ic}: max|resid|={np.max(np.abs(sub2[ic].values-p)):.2e}')
print(f'Changed: {sum(((sub2.loc[~original_nan[c],c]-raw.loc[~original_nan[c],c]).abs()>1e-10).sum() for c in cols)}')
submit_answer_2(sub2)

---
## Problem 3: Minimize Purchase Cost (15%)

Stateless. Formula recovery + interpolation at timestamp. All-in on cheapest.

| Approach | Capture | Why |
|----------|---------|-----|
| **Formula + Interp** | **99%** | Temporal autocorrelation ~0.99 |
| Pairwise Ratio | 46% | Best cross-sectional |
| Kalman Filter | 59% | No cross-column structure |
| HistGBR | 5% | Level drift OOS |
| Cond. Gaussian | -2% | Stale mean |

In [ ]:
import pandas as pd
import numpy as np

_hist = pd.read_csv('limestone_data_challenge_2026.data.csv')
_cols = [c for c in _hist.columns if c.startswith('col_')]
_col_idx = {c: i for i, c in enumerate(_cols)}
_C = len(_cols)
_hist_px = _hist[_cols].values
_N = len(_hist_px)

_interp_px = np.full_like(_hist_px, np.nan)
for _j in range(_C):
    _s = pd.Series(_hist_px[:, _j])
    _interp_px[:, _j] = _s.interpolate(method='linear').ffill().bfill().values
_last_interp = _interp_px[-1].copy()

_formulas = {
    'col_11': {'col_42': 0.307570522963546, 'col_28': 0.342416834876758, 'col_20': 0.212761996386421, 'col_07': 0.074628357326379, 'col_22': 0.062622288446896},
    'col_50': {'col_42': 0.586385343439102, 'col_32': 0.22475351473822, 'col_26': 0.188861141822677},
    'col_48': {'col_05': 0.5453629329, 'col_23': 0.1268823241, 'col_45': 0.1348281464, 'col_04': 0.0968378316, 'col_26': 0.0960887650},
}
_index_cols = ['col_11', 'col_50', 'col_30', 'col_48', 'col_46']

_ratios = np.ones(_C)
for _j in range(_C):
    _r = []
    for _i in range(_N):
        if np.isnan(_hist_px[_i, _j]): continue
        _others = [_hist_px[_i, k] for k in range(_C) if k != _j and not np.isnan(_hist_px[_i, k])]
        if len(_others) < 5: continue
        _r.append(_hist_px[_i, _j] / np.mean(_others))
    _ratios[_j] = np.median(_r) if _r else 1.0

def _formula_recover(vals):
    for _ in range(3):
        for ic, form in _formulas.items():
            ji = _col_idx[ic]; fjs = [_col_idx[f] for f in form.keys()]
            if np.isnan(vals[ji]) and all(not np.isnan(vals[fj]) for fj in fjs):
                vals[ji] = sum(w * vals[_col_idx[f]] for f, w in form.items())
            if not np.isnan(vals[ji]):
                for fc, wc in form.items():
                    fi = _col_idx[fc]
                    if np.isnan(vals[fi]):
                        oth = [(f, w) for f, w in form.items() if f != fc]
                        if all(not np.isnan(vals[_col_idx[f]]) for f, _ in oth):
                            vals[fi] = (vals[ji] - sum(w * vals[_col_idx[f]] for f, w in oth)) / wc
    return vals

def _predict_row(prices):
    row_vals = np.array([prices.get(c, np.nan) if c in prices.index else np.nan for c in _cols])
    _formula_recover(row_vals)
    nan_cols = [_cols[j] for j in range(_C) if np.isnan(prices.get(_cols[j], np.nan))]
    known = {_cols[j]: prices[_cols[j]] for j in range(_C) if not np.isnan(prices.get(_cols[j], np.nan))}
    if not nan_cols: return {}, known
    t = prices.get('time', None)
    t = int(t) if t is not None and not (isinstance(t, float) and np.isnan(t)) else None
    predicted = {}
    for c in nan_cols:
        j = _col_idx[c]
        if not np.isnan(row_vals[j]):
            predicted[c] = row_vals[j]
        elif t is not None and 0 <= t < _N:
            predicted[c] = _interp_px[t, j]
        else:
            obs_mean = np.nanmean([prices.get(x, np.nan) for x in _cols])
            predicted[c] = 0.7 * _last_interp[j] + 0.3 * _ratios[j] * obs_mean if not np.isnan(obs_mean) else _last_interp[j]
    return predicted, known

def trading_problem_3(prices):
    predicted, known = _predict_row(prices)
    nan_cols = list(predicted.keys())
    if not nan_cols:
        nan_cols = [c for c in _cols if pd.isna(prices.get(c, np.nan))]
        if not nan_cols: nan_cols = [_cols[0]]
        return pd.DataFrame({'purchase_col': [nan_cols[0]], 'qty': [100]})
    cheapest = min(nan_cols, key=lambda c: predicted[c])
    return pd.DataFrame({'purchase_col': [cheapest], 'qty': [100]})

validate_trading_problem_3(trading_problem_3)

---
## Problem 4: Maximize Arbitrage Profit (20%)

Stateless. Best (NaN_src → index_dest) spread. All-in 100kg or trade 0.

Observed index destinations have zero prediction error. 98% oracle capture.

In [ ]:
import pandas as pd
import numpy as np

_hist = pd.read_csv('limestone_data_challenge_2026.data.csv')
_cols = [c for c in _hist.columns if c.startswith('col_')]
_col_idx = {c: i for i, c in enumerate(_cols)}
_C = len(_cols)
_hist_px = _hist[_cols].values
_N = len(_hist_px)

_interp_px = np.full_like(_hist_px, np.nan)
for _j in range(_C):
    _s = pd.Series(_hist_px[:, _j])
    _interp_px[:, _j] = _s.interpolate(method='linear').ffill().bfill().values
_last_interp = _interp_px[-1].copy()

_formulas = {
    'col_11': {'col_42': 0.307570522963546, 'col_28': 0.342416834876758, 'col_20': 0.212761996386421, 'col_07': 0.074628357326379, 'col_22': 0.062622288446896},
    'col_50': {'col_42': 0.586385343439102, 'col_32': 0.22475351473822, 'col_26': 0.188861141822677},
    'col_48': {'col_05': 0.5453629329, 'col_23': 0.1268823241, 'col_45': 0.1348281464, 'col_04': 0.0968378316, 'col_26': 0.0960887650},
}
_index_cols = ['col_11', 'col_50', 'col_30', 'col_48', 'col_46']

_ratios = np.ones(_C)
for _j in range(_C):
    _r = []
    for _i in range(_N):
        if np.isnan(_hist_px[_i, _j]): continue
        _others = [_hist_px[_i, k] for k in range(_C) if k != _j and not np.isnan(_hist_px[_i, k])]
        if len(_others) < 5: continue
        _r.append(_hist_px[_i, _j] / np.mean(_others))
    _ratios[_j] = np.median(_r) if _r else 1.0

def _formula_recover(vals):
    for _ in range(3):
        for ic, form in _formulas.items():
            ji = _col_idx[ic]; fjs = [_col_idx[f] for f in form.keys()]
            if np.isnan(vals[ji]) and all(not np.isnan(vals[fj]) for fj in fjs):
                vals[ji] = sum(w * vals[_col_idx[f]] for f, w in form.items())
            if not np.isnan(vals[ji]):
                for fc, wc in form.items():
                    fi = _col_idx[fc]
                    if np.isnan(vals[fi]):
                        oth = [(f, w) for f, w in form.items() if f != fc]
                        if all(not np.isnan(vals[_col_idx[f]]) for f, _ in oth):
                            vals[fi] = (vals[ji] - sum(w * vals[_col_idx[f]] for f, w in oth)) / wc
    return vals

def _predict_row(prices):
    row_vals = np.array([prices.get(c, np.nan) if c in prices.index else np.nan for c in _cols])
    _formula_recover(row_vals)
    nan_cols = [_cols[j] for j in range(_C) if np.isnan(prices.get(_cols[j], np.nan))]
    known = {_cols[j]: prices[_cols[j]] for j in range(_C) if not np.isnan(prices.get(_cols[j], np.nan))}
    if not nan_cols: return {}, known
    t = prices.get('time', None)
    t = int(t) if t is not None and not (isinstance(t, float) and np.isnan(t)) else None
    predicted = {}
    for c in nan_cols:
        j = _col_idx[c]
        if not np.isnan(row_vals[j]):
            predicted[c] = row_vals[j]
        elif t is not None and 0 <= t < _N:
            predicted[c] = _interp_px[t, j]
        else:
            obs_mean = np.nanmean([prices.get(x, np.nan) for x in _cols])
            predicted[c] = 0.7 * _last_interp[j] + 0.3 * _ratios[j] * obs_mean if not np.isnan(obs_mean) else _last_interp[j]
    return predicted, known

def trading_problem_4(prices):
    predicted, known = _predict_row(prices)
    nan_cols = list(predicted.keys())
    if not nan_cols:
        return pd.DataFrame({'src_col': pd.Series(dtype=str), 'dest_col': pd.Series(dtype=str), 'qty': pd.Series(dtype=int)})
    dest_prices = {}
    for ic in _index_cols:
        if ic in known: dest_prices[ic] = known[ic]
        elif ic in predicted: dest_prices[ic] = predicted[ic]
    best_spread = 0; best_src = best_dest = None
    for src in nan_cols:
        if src not in predicted: continue
        for dest, dest_px in dest_prices.items():
            if src == dest: continue
            spread = dest_px - predicted[src]
            if spread > best_spread:
                best_spread = spread; best_src, best_dest = src, dest
    if best_src is None or best_spread <= 0:
        return pd.DataFrame({'src_col': pd.Series(dtype=str), 'dest_col': pd.Series(dtype=str), 'qty': pd.Series(dtype=int)})
    return pd.DataFrame({'src_col': [best_src], 'dest_col': [best_dest], 'qty': [100]})

validate_trading_problem_4(trading_problem_4)

---
## Problem 5: Limit Orders vs Median (20%)

Stateless. Bid = predicted × 1.01. All-in on largest (median − bid) gap.

| Buffer | Capture | Fill% |
|--------|---------|-------|
| 0% | 60% | 77% |
| **1%** | **66%** | **99.9%** |
| 2% | 53% | 99.9% |
| 5% | 15% | 100% |

87% oracle capture.

In [ ]:
import pandas as pd
import numpy as np

_hist = pd.read_csv('limestone_data_challenge_2026.data.csv')
_cols = [c for c in _hist.columns if c.startswith('col_')]
_col_idx = {c: i for i, c in enumerate(_cols)}
_C = len(_cols)
_hist_px = _hist[_cols].values
_N = len(_hist_px)

_interp_px = np.full_like(_hist_px, np.nan)
for _j in range(_C):
    _s = pd.Series(_hist_px[:, _j])
    _interp_px[:, _j] = _s.interpolate(method='linear').ffill().bfill().values
_last_interp = _interp_px[-1].copy()

_formulas = {
    'col_11': {'col_42': 0.307570522963546, 'col_28': 0.342416834876758, 'col_20': 0.212761996386421, 'col_07': 0.074628357326379, 'col_22': 0.062622288446896},
    'col_50': {'col_42': 0.586385343439102, 'col_32': 0.22475351473822, 'col_26': 0.188861141822677},
    'col_48': {'col_05': 0.5453629329, 'col_23': 0.1268823241, 'col_45': 0.1348281464, 'col_04': 0.0968378316, 'col_26': 0.0960887650},
}
_index_cols = ['col_11', 'col_50', 'col_30', 'col_48', 'col_46']

_ratios = np.ones(_C)
for _j in range(_C):
    _r = []
    for _i in range(_N):
        if np.isnan(_hist_px[_i, _j]): continue
        _others = [_hist_px[_i, k] for k in range(_C) if k != _j and not np.isnan(_hist_px[_i, k])]
        if len(_others) < 5: continue
        _r.append(_hist_px[_i, _j] / np.mean(_others))
    _ratios[_j] = np.median(_r) if _r else 1.0

def _formula_recover(vals):
    for _ in range(3):
        for ic, form in _formulas.items():
            ji = _col_idx[ic]; fjs = [_col_idx[f] for f in form.keys()]
            if np.isnan(vals[ji]) and all(not np.isnan(vals[fj]) for fj in fjs):
                vals[ji] = sum(w * vals[_col_idx[f]] for f, w in form.items())
            if not np.isnan(vals[ji]):
                for fc, wc in form.items():
                    fi = _col_idx[fc]
                    if np.isnan(vals[fi]):
                        oth = [(f, w) for f, w in form.items() if f != fc]
                        if all(not np.isnan(vals[_col_idx[f]]) for f, _ in oth):
                            vals[fi] = (vals[ji] - sum(w * vals[_col_idx[f]] for f, w in oth)) / wc
    return vals

def _predict_row(prices):
    row_vals = np.array([prices.get(c, np.nan) if c in prices.index else np.nan for c in _cols])
    _formula_recover(row_vals)
    nan_cols = [_cols[j] for j in range(_C) if np.isnan(prices.get(_cols[j], np.nan))]
    known = {_cols[j]: prices[_cols[j]] for j in range(_C) if not np.isnan(prices.get(_cols[j], np.nan))}
    if not nan_cols: return {}, known
    t = prices.get('time', None)
    t = int(t) if t is not None and not (isinstance(t, float) and np.isnan(t)) else None
    predicted = {}
    for c in nan_cols:
        j = _col_idx[c]
        if not np.isnan(row_vals[j]):
            predicted[c] = row_vals[j]
        elif t is not None and 0 <= t < _N:
            predicted[c] = _interp_px[t, j]
        else:
            obs_mean = np.nanmean([prices.get(x, np.nan) for x in _cols])
            predicted[c] = 0.7 * _last_interp[j] + 0.3 * _ratios[j] * obs_mean if not np.isnan(obs_mean) else _last_interp[j]
    return predicted, known

def trading_problem_5(prices):
    predicted, known = _predict_row(prices)
    nan_cols = list(predicted.keys())
    if not nan_cols:
        nan_cols = [c for c in _cols if pd.isna(prices.get(c, np.nan))]
        if not nan_cols:
            return pd.DataFrame({'order_col': pd.Series(dtype=str), 'px': pd.Series(dtype=float), 'qty': pd.Series(dtype=int)})
    all_prices = list(known.values()) + [predicted[c] for c in nan_cols]
    est_median = np.median(all_prices)
    buffer = 0.01
    candidates = []
    for c in nan_cols:
        if c not in predicted: continue
        bid = predicted[c] * (1 + buffer)
        if bid < est_median:
            candidates.append((c, bid, est_median - bid))
    if not candidates:
        cheapest = min(nan_cols, key=lambda c: predicted.get(c, float('inf')))
        bid = predicted.get(cheapest, est_median * 0.95) * (1 + buffer)
        candidates = [(cheapest, bid, max(0, est_median - bid))]
    best = max(candidates, key=lambda x: x[2])
    return pd.DataFrame({'order_col': [best[0]], 'px': [best[1]], 'qty': [100]})

validate_trading_problem_5(trading_problem_5)

---
## Methodology Notes

### Constraint Propagation
col_50's formula → derive col_26 → unlock col_48 verification on 18 rows (previously 0). Algebraic substitution confirms on 17 raw rows at RMSE = 3×10⁻¹⁰.

### Why Simple Beats Complex
53 columns >93% correlated. Daily autocorrelation ~0.99. Median NaN gap = 1 row. Interpolation is near-optimal; ML adds noise.

Tested: HistGBR, Kalman, Conditional Gaussian, PCA reconstruction, pairwise ratios, Ridge on deviations, ensembles. None beat interpolation.

### Robustness
- **Stateless** (per FAQ: no global state across calls)
- Edge cases: 0 NaN, all NaN, out-of-range timestamps
- OOS fallback: 70% last_interp + 30% ratio × row_mean